# 03 — Portable records and leakage-safe splits

        **Estimated time:** 40 minutes<br>
        **Prerequisites:** 02 — Dataset exploration and validation<br>
        **Learner-produced evidence:** split counts, stable IDs, hashes, and a zero-leakage report

        ## Learning objectives

        - Trace immutable source rows into framework-neutral chat records.
- Explain group-aware balancing and stable record identifiers.
- Verify split integrity without inspecting frozen-test examples.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Evidence boundaries

Train fits weights and train-derived rules. Validation selects prompts
and settings. The frozen test is opened only after those choices stop.
This notebook reads its manifest and runs automated leakage checks, but
does not display or use test examples.


In [ ]:
import json

from aai_local_finetuning.data import check_split_files, text_similarity
from aai_local_finetuning.evaluation import load_records_jsonl
from aai_local_finetuning.settings import load_settings

settings = load_settings()
processed = settings.processed_dir
manifest = json.loads((processed / "manifest.json").read_text(encoding="utf-8"))
train = tuple(load_records_jsonl(processed / "train.jsonl"))
validation = tuple(load_records_jsonl(processed / "valid.jsonl"))
split_contract = {
    name: {
        "records": descriptor["record_count"],
        "frozen": descriptor["frozen"],
        "sha256": descriptor["sha256"],
    }
    for name, descriptor in manifest["splits"].items()
}
split_contract

## What one portable record contains

Framework-neutral JSONL preserves identity, source version, messages,
labels, grouping evidence, flags, and difficulty. Source responses are
not copied as training targets; a versioned response policy renders a
short target independently. The preview is already masked and bounded.


In [ ]:
example = train[0]
safe_preview = {
    "example_id": example.example_id,
    "input_preview": example.input_text[:160],
    "target": {
        "intent": example.target.intent,
        "category": example.target.category,
        "requires_escalation": example.target.requires_escalation,
        "response_preview": example.target.response[:100],
    },
    "flags": example.flags,
    "difficulty": example.difficulty,
    "split_group": example.metadata.get("split_group"),
}
safe_preview

## Balance and separation

The source has no reliable conversation, account, document, template,
or timestamp identifier. The pipeline therefore keeps inferred exact,
template, and near-duplicate groups together, excludes label-conflicting
groups, and records this limitation instead of inventing source fields.


In [ ]:
train_intents = {}
validation_intents = {}
for record in train:
    train_intents[record.target.intent] = train_intents.get(record.target.intent, 0) + 1
for record in validation:
    validation_intents[record.target.intent] = (
        validation_intents.get(record.target.intent, 0) + 1
    )
balance = {
    "train_unique_counts": sorted(set(train_intents.values())),
    "validation_unique_counts": sorted(set(validation_intents.values())),
    "train_validation_id_overlap": len(
        {item.example_id for item in train} & {item.example_id for item in validation}
    ),
    "dataset_fingerprint": manifest["dataset_fingerprint"],
}
balance

## Run the automated leakage gate

A passing gate means no configured relationship crossed the prepared
boundaries. It does not prove that heuristic grouping discovered every
semantic relationship in the world.


In [ ]:
leakage = check_split_files(processed)
{
    "passed": leakage.passed,
    "finding_count": len(leakage.findings),
    "findings_by_kind": leakage.counts,
}

## Exercise — reason about near duplicates

Change the second invented sentence and observe the similarity. Decide
whether a numeric threshold is sufficient evidence of common origin.
Success means you record both a decision and a limitation.


In [ ]:
sentence_a = "I forgot my password and cannot sign in"
sentence_b = "I cannot sign in because I forgot the password"
similarity = text_similarity(sentence_a, sentence_b)
threshold = 0.88
grouping_decision = similarity >= threshold
limitation = (
    "Similarity is a reproducible heuristic; it is not a verified "
    "conversation or template identifier from the source."
)
{
    "similarity": round(similarity, 3),
    "same_group_at_threshold": grouping_decision,
    "limitation": limitation,
}

**Hint:** a threshold makes behavior repeatable, not automatically true.
Group conservatively and preserve the documented uncertainty.


## Checkpoint

You can identify what each split is allowed to influence and explain
why a frozen flag, file hash, stable IDs, and leakage test work together.

**Next:** `04_deterministic_baselines.ipynb` establishes a sanity floor
and a transparent meaningful baseline using train and validation only.
